In [4]:
import pandas as pd
import sqlite3

In [6]:
df = pd.read_csv("Desktop/internshipproject.csv")

df.head()

,Timestamp,MachineID,Plant,Temperature,Vibration,Pressure,EnergyConsumption,ProductionUnits,DefectCount,MaintenanceFlag,...,vib_anomaly,press_mean,press_std,p upper limit,p lower limit,press_anomaly,temp_z,Vib_z,press_z,condition_index
0,2025-01-01 00:00:00,135,Plant_B,70.368090,5.924252,30.146306,187.902319,158,2,0,...,Normal,29.995756,4.603488,43.806220,16.185292,Normal,0.106124,0.675688,0.032704,0.254967
1,2025-01-01 01:00:00,114,Plant_A,59.632304,5.408591,24.651498,270.420824,170,3,0,...,Normal,29.843421,5.268647,45.649363,14.037480,Normal,-1.041843,0.257794,-0.985438,0.789707
2,2025-01-01 02:00:00,147,Plant_C,71.513375,4.277121,33.922249,205.519258,68,2,1,...,Normal,30.166958,4.630228,44.057642,16.276274,Normal,0.066783,-0.466805,0.811038,0.410066
3,2025-01-01 03:00:00,138,Plant_B,84.742846,4.013507,37.431346,177.003242,75,3,0,...,Normal,29.866242,5.037829,44.979729,14.752754,Normal,1.638255,-0.586569,1.501660,1.281771
4,2025-01-01 04:00:00,116,Plant_C,61.904598,3.402422,24.758312,272.724353,187,4,0,...,Normal,30.022429,5.145782,45.459775,14.585083,Normal,-0.723708,-1.086349,-1.022997,0.922287


In [8]:
conn = sqlite3.connect("sensor.db")
df.to_sql("sensor_data", conn, if_exists="replace", index=False)

3456

In [9]:
query = """
WITH stats AS (
    SELECT 
        Plant,
        MachineID,
        AVG(Temperature) AS temp_mean,
        AVG(Vibration) AS vib_mean,
        AVG(Pressure) AS press_mean
    FROM sensor_data
    GROUP BY Plant, MachineID
),

anomaly_detection AS (
    SELECT 
        d.Plant,
        d.MachineID,
        
        CASE 
        WHEN d.Temperature > s.temp_mean + 3 
        OR d.Temperature < s.temp_mean - 3
        THEN 1 ELSE 0 END AS temp_anomaly,

        CASE 
        WHEN d.Vibration > s.vib_mean + 3 
        OR d.Vibration < s.vib_mean - 3
        THEN 1 ELSE 0 END AS vib_anomaly,

        CASE 
        WHEN d.Pressure > s.press_mean + 3 
        OR d.Pressure < s.press_mean - 3
        THEN 1 ELSE 0 END AS press_anomaly

    FROM sensor_data d
    JOIN stats s
    ON d.Plant = s.Plant
    AND d.MachineID = s.MachineID
)

SELECT 
Plant,
MachineID,
SUM(temp_anomaly) AS temp_anomaly_count,
SUM(vib_anomaly) AS vib_anomaly_count,
SUM(press_anomaly) AS press_anomaly_count
FROM anomaly_detection
GROUP BY Plant, MachineID
ORDER BY Plant, MachineID;
"""

In [10]:
result = pd.read_sql_query(query, conn)

result

,Plant,MachineID,temp_anomaly_count,vib_anomaly_count,press_anomaly_count
0,Plant_A,100,20,0,12
1,Plant_A,101,23,0,12
2,Plant_A,102,11,1,5
3,Plant_A,103,21,1,18
4,Plant_A,104,13,1,9
...,...,...,...,...,...
145,Plant_C,145,17,0,11
146,Plant_C,146,13,1,12
147,Plant_C,147,20,1,14
148,Plant_C,148,16,0,16


In [11]:
result.to_csv("anomaly_counts.csv", index=False)

In [14]:
create_view_query = """
CREATE VIEW sensor_condition_2025 AS
SELECT 
MachineID,
AVG(Temperature) AS AvgTemp,
AVG(Vibration) AS AvgVib,
AVG(Pressure) AS AvgPressure,
AVG(EnergyConsumption/ProductionUnits) AS AvgEnergyPerUnit,
SUM(DefectCount) AS TotalDefects,
SUM(MaintenanceFlag) AS MaintenanceCount,

CASE 
    WHEN SUM(DefectCount) >= (
        SELECT MAX(TotalDefects)*0.9
        FROM (
            SELECT SUM(DefectCount) AS TotalDefects
            FROM sensor_data
            GROUP BY MachineID
        )
    )
    THEN 'Critical'
    ELSE 'Normal'
END AS ConditionBucket

FROM sensor_data
GROUP BY MachineID;
"""

In [15]:
conn.execute(create_view_query)

In [16]:
view_result = pd.read_sql_query(
"SELECT * FROM sensor_condition_2025",
conn
)

view_result.head()

,MachineID,AvgTemp,AvgVib,AvgPressure,AvgEnergyPerUnit,TotalDefects,MaintenanceCount,ConditionBucket
0,100,69.461851,5.230727,29.971427,2.551883,212,8,Normal
1,101,70.587752,4.998503,29.415142,2.335212,247,10,Critical
2,102,67.197634,4.818878,30.311753,2.279829,193,7,Normal
3,103,70.161168,5.024913,30.170780,2.377382,258,9,Critical
4,104,71.768228,4.941795,30.352676,2.380145,176,4,Normal


In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest

In [4]:
df = pd.read_csv("Desktop/internshipproject.csv")

df.head()

,Timestamp,MachineID,Plant,Temperature,Vibration,Pressure,EnergyConsumption,ProductionUnits,DefectCount,MaintenanceFlag,...,vib_anomaly,press_mean,press_std,p upper limit,p lower limit,press_anomaly,temp_z,Vib_z,press_z,condition_index
0,2025-01-01 00:00:00,135,Plant_B,70.368090,5.924252,30.146306,187.902319,158,2,0,...,Normal,29.995756,4.603488,43.806220,16.185292,Normal,0.106124,0.675688,0.032704,0.254967
1,2025-01-01 01:00:00,114,Plant_A,59.632304,5.408591,24.651498,270.420824,170,3,0,...,Normal,29.843421,5.268647,45.649363,14.037480,Normal,-1.041843,0.257794,-0.985438,0.789707
2,2025-01-01 02:00:00,147,Plant_C,71.513375,4.277121,33.922249,205.519258,68,2,1,...,Normal,30.166958,4.630228,44.057642,16.276274,Normal,0.066783,-0.466805,0.811038,0.410066
3,2025-01-01 03:00:00,138,Plant_B,84.742846,4.013507,37.431346,177.003242,75,3,0,...,Normal,29.866242,5.037829,44.979729,14.752754,Normal,1.638255,-0.586569,1.501660,1.281771
4,2025-01-01 04:00:00,116,Plant_C,61.904598,3.402422,24.758312,272.724353,187,4,0,...,Normal,30.022429,5.145782,45.459775,14.585083,Normal,-0.723708,-1.086349,-1.022997,0.922287


In [5]:
features = df[[
    "Temperature",
    "Vibration",
    "Pressure",
    "EnergyConsumption",
    "ProductionUnits"
]]

In [6]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

df["anomaly"] = model.fit_predict(features)

In [7]:
anomalies = df[df["anomaly"] == -1]

top_200 = anomalies[["Timestamp","MachineID"]].head(200)

top_200

,Timestamp,MachineID
19,2025-01-01 19:00:00,127
34,2025-01-02 10:00:00,118
89,2025-01-04 17:00:00,131
114,2025-01-05 18:00:00,118
129,2025-01-06 09:00:00,113
...,...,...
3319,2025-05-19 07:00:00,130
3355,2025-05-20 19:00:00,118
3368,2025-05-21 08:00:00,102
3379,2025-05-21 19:00:00,114


In [8]:
top_200.to_csv("top_200_anomalies.csv", index=False)

In [9]:
df["HealthScore"] = (
100
- 0.25*df["Temperature"]
- 0.25*df["Vibration"]
- 0.20*df["Pressure"]
- 0.15*df["EnergyConsumption"]
- 0.10*df["ProductionUnits"]
- 0.05*df["DefectCount"]
)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [11]:
X = df[[
"Temperature",
"Vibration",
"Pressure",
"EnergyConsumption",
"ProductionUnits",
"DefectCount"
]]

y = df["HealthScore"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [13]:
model = LinearRegression()

model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [14]:
predictions = model.predict(X_test)

In [15]:
from sklearn.metrics import r2_score

r2_score(y_test, predictions)

1.0

In [16]:
importance = pd.DataFrame({
"Feature": X.columns,
"Coefficient": model.coef_
})

importance

,Feature,Coefficient
0,Temperature,-0.25
1,Vibration,-0.25
2,Pressure,-0.20
3,EnergyConsumption,-0.15
4,ProductionUnits,-0.10
5,DefectCount,-0.05
